# Phase 2 Data Pipeline Validation

This notebook validates the reproducible Phase 2 data/metrics pipeline for the qBraid / JonesTrading Track A volatility-regime project.

Scope for this notebook:

- load the processed SPY+VIX fallback dataset;
- verify chronological train/validation/test splits;
- verify train-only transition-threshold construction;
- build leakage-safe normalized regression arrays;
- build sequence arrays for ESN/QRC-style models;
- evaluate the persistence baseline floor using Track A metrics.

This is a data-pipeline validation notebook, not a model-optimization notebook. Classical baseline modeling is handled separately.

## 1. Imports

In [ ]:
import pandas as pd

from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.splits import (
    add_train_only_transition_flags,
    chronological_tabular_split,
    describe_regression_splits,
)
from qpitome_qrc.data.features import (
    FEATURE_COLUMNS,
    make_regression_arrays,
    make_sequence_arrays,
)
from qpitome_qrc.evaluation.metrics import (
    evaluate_volatility_forecast,
    volatility_metrics_to_frame,
)

## 2. Load processed fallback dataset

The executable Phase 2 path uses the processed SPY+VIX fallback dataset. VOLARE can remain a preferred scientific upgrade, but this path is immediately reproducible without gated data access.

In [ ]:
df = load_phase2_volatility_data()

print(f"shape: {df.shape}")
print(f"date range: {df['date'].min()} -> {df['date'].max()}")
df[["future_rv_5d", "future_rv_20d"]].describe(percentiles=[0.25, 0.5, 0.75])

## 3. Validate chronological splits

All splits are chronological. No random shuffling is used for this time-series forecasting task.

In [ ]:
splits = chronological_tabular_split(df)
split_summary = describe_regression_splits(splits)
split_summary

## 4. Validate train-only transition thresholds

Transition flags are used for interpretation and regime-warning analysis. Thresholds are computed from training data only, then applied unchanged to validation/test data.

In [ ]:
df_flags, transition_info = add_train_only_transition_flags(df)

transition_counts = (
    df_flags.groupby("split")["transition_event"]
    .agg(count="count", events="sum")
    .assign(event_rate=lambda x: x["events"] / x["count"])
)

transition_info, transition_counts

## 5. Build leakage-safe normalized regression arrays

Feature normalization is performed after splitting. The scaler is fit on the training split only and applied unchanged to validation and test splits.

In [ ]:
arrays_20d, scaler_20d = make_regression_arrays(
    splits,
    target_column="future_rv_20d",
    scaler_name="standard",
)

array_summary = pd.DataFrame(
    [
        {"split": "train", "X_shape": arrays_20d.X_train.shape, "y_shape": arrays_20d.y_train.shape},
        {"split": "validation", "X_shape": arrays_20d.X_val.shape, "y_shape": arrays_20d.y_val.shape},
        {"split": "test", "X_shape": arrays_20d.X_test.shape, "y_shape": arrays_20d.y_test.shape},
    ]
)

print(f"feature_count: {len(arrays_20d.feature_columns)}")
print(f"target: {arrays_20d.target_column}")
print(f"scaler: {arrays_20d.scaler_name}")
array_summary

## 6. Build sequence arrays for reservoir models

A 20-trading-day lookback provides model-ready sequence arrays for ESN/QRC-style reservoirs. The target remains the forward-looking realized-volatility target already constructed in the processed dataset.

In [ ]:
X_seq, y_seq, seq_dates = make_sequence_arrays(
    splits["train"],
    target_column="future_rv_20d",
    lookback=20,
)

print(f"X_train_sequence: {X_seq.shape}")
print(f"y_train_sequence: {y_seq.shape}")
print(f"first_sequence_target_date: {seq_dates.iloc[0]}")
print(f"last_sequence_target_date: {seq_dates.iloc[-1]}")
print(f"sequence_count_check: {len(splits['train'])} - 20 + 1 = {len(splits['train']) - 20 + 1}")

## 7. Evaluate persistence baseline floor

Persistence forecasts are the first baseline floor: current trailing realized volatility is used as the forecast for future realized volatility.

In [ ]:
baseline_specs = [
    ("persistence_5d_to_5d", "future_rv_5d", "rv_5d"),
    ("persistence_10d_to_5d", "future_rv_5d", "rv_10d"),
    ("persistence_20d_to_20d", "future_rv_20d", "rv_20d"),
    ("persistence_60d_to_20d", "future_rv_20d", "rv_60d"),
]

rows = []
for split_name, split_df in splits.items():
    for model_name, target_col, predictor_col in baseline_specs:
        metrics = evaluate_volatility_forecast(
            split_df[target_col].to_numpy(),
            split_df[predictor_col].to_numpy(),
        )
        row = volatility_metrics_to_frame(metrics).iloc[0].to_dict()
        row.update(
            {
                "split": split_name,
                "model": model_name,
                "target": target_col,
                "predictor": predictor_col,
                "n": len(split_df),
            }
        )
        rows.append(row)

baseline_table = pd.DataFrame(rows)
baseline_table = baseline_table[
    ["split", "model", "target", "predictor", "n", "rmse", "qlike", "mz_alpha", "mz_beta", "mz_r2"]
]
baseline_table.sort_values(["target", "split", "rmse"])

## 8. May 22 sign-off summary

In [ ]:
signoff = {
    "processed_rows": len(df),
    "processed_columns": df.shape[1],
    "feature_count": len(FEATURE_COLUMNS),
    "targets": ["future_rv_5d", "future_rv_20d"],
    "train_rows": len(splits["train"]),
    "validation_rows": len(splits["val"]),
    "test_rows": len(splits["test"]),
    "tabular_train_shape": arrays_20d.X_train.shape,
    "sequence_train_shape": X_seq.shape,
    "track_a_metrics": ["RMSE", "QLIKE", "Mincer-Zarnowitz"],
}
signoff